<a href="https://colab.research.google.com/github/hfelizzola/Investigaciones-de-Operaciones-I/blob/main/programacion-lineal/01_problema_produccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Tutorial: Modelo de Producción con GAMSPy

# Investigación de Operaciones I: Modelado en GAMSPY
## Ejercicio 2: Planificación de Producción con Penalizaciones (Un Solo Periodo)

### Enunciado del Problema

**(Taha, 2012)** Para la temporada de fin de año, una fábrica de ropa está produciendo abrigos con capota, chaquetas de plumas, pantalones térmicos y guantes. La producción pasa por cuatro departamentos: corte, aislamiento, costura y empaque. La fábrica ya tiene pedidos fijos de sus clientes. El contrato establece una multa (penalización) por cada prenda no entregada. El objetivo es crear un plan de producción que **maximice la ganancia**.

**Datos de Producción y Costos**

| Departamento | Abrigos | Chaquetas de Plumas | Pantalones Térmicos | Guantes | Capacidad (h) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Corte** | 0.30 | 0.30 | 0.25 | 0.15 | 1000 |
| **Aislamiento** | 0.25 | 0.35 | 0.30 | 0.10 | 1000 |
| **Costura** | 0.45 | 0.50 | 0.40 | 0.22 | 1000 |
| **Empaque** | 0.15 | 0.15 | 0.10 | 0.05 | 1000 |
| **Demanda (unidades)** | 800 | 750 | 600 | 500 | |
| **Utilidad (\$)** | 30 | 40 | 20 | 10 | |
| **Penalización (\$)** | 15 | 20 | 10 | 8 | |

---

### Modelo Matemático

#### 1. Conjuntos e Índices
* $I$: Departamentos, indexados por $i \in \{\text{Corte}, \text{Aislamiento}, \text{Costura}, \text{Empaque}\}$.
* $J$: Productos, indexados por $j \in \{\text{Abrigos}, \text{Chaquetas}, \text{Pantalones}, \text{Guantes}\}$.

#### 2. Parámetros
* $CA_i$: Capacidad en horas del departamento $i$.
* $DE_j$: Demanda del producto $j$.
* $UT_j$: Utilidad unitaria del producto $j$.
* $PE_j$: Penalización por cada unidad faltante del producto $j$.
* $TP_{ij}$: Tiempo de procesamiento del producto $j$ en el departamento $i$.

#### 3. Variables de Decisión
* $x_j \geq 0$: Cantidad a fabricar del producto $j$.
* $s_j \geq 0$: Faltantes (unidades no entregadas o penalizadas) del producto $j$.

#### 4. Función Objetivo
Maximizar la utilidad total (ganancia por ventas menos multa por faltantes):
$$\text{Max } Z = \sum_{j \in J} \left( UT_j \cdot x_j - PE_j \cdot s_j \right)$$

#### 5. Restricciones

**R1. Capacidad de Departamentos:** El tiempo total usado en cada departamento no puede superar su capacidad.
$$\sum_{j \in J} TP_{ij} \cdot x_j \leq CA_i \quad \forall i \in I$$

**R2. Balance de Demanda:** Lo fabricado más lo que falta por entregar debe ser igual a la demanda total exigida.
$$x_j + s_j = DE_j \quad \forall j \in J$$

**R3. No Negatividad:**
$$x_j, s_j \geq 0 \quad \forall j \in J$$

## Instalación de la librería GAMSPY

In [ ]:
# Instalar GAMSPY (ejecutar solo una vez por entorno)
!pip install gamspy -q

import gamspy as gp



In [ ]:
# Crear el contenedor principal
m = gp.Container()

# Definir Conjuntos
i = gp.Set(m, name="i", records=["Corte", "Aislamiento", "Costura", "Empaque"], description="Departamentos")
j = gp.Set(m, name="j", records=["Abrigos", "Chaquetas", "Pantalones", "Guantes"], description="Productos")

print("Contenedor y conjuntos creados.")

Contenedor y conjuntos creados.


**Carga de Parámetros**

Aquí introducimos un parámetro que depende de dos índices (departamento y producto): el tiempo de procesamiento $TP_{ij}$. Usaremos listas de tuplas para mapear estos datos de forma clara.

In [ ]:
# Capacidad CA(i)
datos_capacidad = [("Corte", 1000), ("Aislamiento", 1000), ("Costura", 1000), ("Empaque", 1000)]
CA = gp.Parameter(m, name="CA", domain=[i], records=datos_capacidad)

# Demanda DE(j)
datos_demanda = [("Abrigos", 800), ("Chaquetas", 750), ("Pantalones", 600), ("Guantes", 500)]
DE = gp.Parameter(m, name="DE", domain=[j], records=datos_demanda)

# Utilidad UT(j)
datos_utilidad = [("Abrigos", 30), ("Chaquetas", 40), ("Pantalones", 20), ("Guantes", 10)]
UT = gp.Parameter(m, name="UT", domain=[j], records=datos_utilidad)

# Penalizacion PE(j)
datos_penalizacion = [("Abrigos", 15), ("Chaquetas", 20), ("Pantalones", 10), ("Guantes", 8)]
PE = gp.Parameter(m, name="PE", domain=[j], records=datos_penalizacion)

# Tiempos de Procesamiento TP(i, j)
datos_tiempos = [
    # Tiempos en Corte
    ("Corte", "Abrigos", 0.30), ("Corte", "Chaquetas", 0.30), ("Corte", "Pantalones", 0.25), ("Corte", "Guantes", 0.15),
    # Tiempos en Aislamiento
    ("Aislamiento", "Abrigos", 0.25), ("Aislamiento", "Chaquetas", 0.35), ("Aislamiento", "Pantalones", 0.30), ("Aislamiento", "Guantes", 0.10),
    # Tiempos en Costura
    ("Costura", "Abrigos", 0.45), ("Costura", "Chaquetas", 0.50), ("Costura", "Pantalones", 0.40), ("Costura", "Guantes", 0.22),
    # Tiempos en Empaque
    ("Empaque", "Abrigos", 0.15), ("Empaque", "Chaquetas", 0.15), ("Empaque", "Pantalones", 0.10), ("Empaque", "Guantes", 0.05)
]
TP = gp.Parameter(m, name="TP", domain=[i, j], records=datos_tiempos)

print("Parámetros cargados exitosamente.")

Parámetros cargados exitosamente.


**Declaración del Modelo Algebraico**

Procedemos a declarar nuestras variables, incluyendo la nueva variable de faltantes $s_j$, y a construir las ecuaciones.

In [ ]:
# 1. Variables de Decisión
x = gp.Variable(m, name="x", domain=[j], type="Positive", description="Unidades producidas")
s = gp.Variable(m, name="s", domain=[j], type="Positive", description="Unidades faltantes (penalizadas)")
Z = gp.Variable(m, name="Z", type="Free", description="Ganancia Neta Total")

# 2. Ecuaciones
# Función Objetivo: Maximizar (Ventas - Multas)
funcion_objetivo = gp.Equation(m, name="funcion_objetivo")
funcion_objetivo[...] = Z == gp.Sum(j, UT[j] * x[j] - PE[j] * s[j])

# Restricción de Capacidad Departamental
capacidad = gp.Equation(m, name="capacidad", domain=[i])
capacidad[i] = gp.Sum(j, TP[i, j] * x[j]) <= CA[i]

# Restricción de Cumplimiento de Demanda
balance_demanda = gp.Equation(m, name="balance_demanda", domain=[j])
balance_demanda[j] = x[j] + s[j] == DE[j]

# 3. Ensamblar y Resolver el Modelo
modelo_ropa = gp.Model(
    m,
    name="Maximizacion_Ganancias",
    equations=m.getEquations(),
    problem="LP",
    sense=gp.Sense.MAX,
    objective=Z
)

modelo_ropa.solve()

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,64625.0,9,9,LP,CPLEX,0.001


**Mostrar Resultados del Modelo**

In [ ]:
# --- Mostrar Resultados ---
print("\n" + "="*50)
print(f"ESTADO DE LA SOLUCIÓN: {modelo_ropa.status}")
print(f"GANANCIA NETA MÁXIMA (Z): ${Z.records.level[0]:,.2f}")
print("="*50 + "\n")

print("--- PLAN DE PRODUCCIÓN (x_j) ---")
print(x.records[['j', 'level']].rename(columns={'j': 'Producto', 'level': 'Fabricar'}).to_string(index=False))

print("\n--- FALTANTES ACEPTADOS (s_j) ---")
df_faltantes = s.records[s.records['level'] > 0]
if not df_faltantes.empty:
    print(df_faltantes[['j', 'level']].rename(columns={'j': 'Producto', 'level': 'Penalizados'}).to_string(index=False))
else:
    print("¡Excelente! Se cumplió con toda la demanda sin penalizaciones.")


ESTADO DE LA SOLUCIÓN: ModelStatus.OptimalGlobal
GANANCIA NETA MÁXIMA (Z): $64,625.00

--- PLAN DE PRODUCCIÓN (x_j) ---
  Producto  Fabricar
   Abrigos     800.0
 Chaquetas     750.0
Pantalones     387.5
   Guantes     500.0

--- FALTANTES ACEPTADOS (s_j) ---
  Producto  Penalizados
Pantalones        212.5


## Análisis de la solución

Con este plan de producción el la utilidad total es de **$64.625**

La siguiente nos muestra la solución de la variable $x$, lo que nos indica que el plan de producción es:

- **800 abrigos**
- **750 Chaquetas**
- **387.5 Pantalones**
- **500 Guantes**



In [ ]:
# Solución de la variable x
x.records

,j,level,marginal,lower,upper,scale
0,Abrigos,800.0,0.0,0.0,inf,1.0
1,Chaquetas,750.0,0.0,0.0,inf,1.0
2,Pantalones,387.5,0.0,0.0,inf,1.0
3,Guantes,500.0,0.0,0.0,inf,1.0


Tambien podemos ver la solución de la variable $s$. En esta tabla se puede observar que:

- No se generan faltantes en los abrigos, chaquetas y guantes.
- Se generan 212.5 faltantes de pantalones.

In [ ]:
# Solución de la variable s
s.records

,j,level,marginal,lower,upper,scale
0,Abrigos,0.0,-11.25,0.0,inf,1.0
1,Chaquetas,0.0,-22.50,0.0,inf,1.0
2,Pantalones,212.5,0.00,0.0,inf,1.0
3,Guantes,0.0,-1.50,0.0,inf,1.0


Tambien podemos hacer una análisis de la capacidad usada. De la siguiente tabla podemos concluir que:

- **Corte:** se utilizan 636,875 horas de las 1000 disponibles.
- **Aislamiento:** se utilizan 628,750 horas de las 1000 disponibles.
- **Costura:** se utilizan 1000 horas de las 1000 disponibles. Por tanto, es un recurso restrictivo.
- **Empaque:** se utilizan 296,250 horas de las 100 disponibles.

In [ ]:
capacidad.records

,i,level,marginal,lower,upper,scale
0,Corte,636.875,0.0,-inf,1000.0,1.0
1,Aislamiento,628.750,0.0,-inf,1000.0,1.0
2,Costura,1000.000,75.0,-inf,1000.0,1.0
3,Empaque,296.250,0.0,-inf,1000.0,1.0


## Modelo Explicito

In [ ]:
m = gp.Container()

# Paso 1: Definir conjuntos (solo para organización, no se usan en las ecuaciones)
productos = gp.Set(m, name="productos", records=[1, 2, 3, 4])
departamentos = gp.Set(m, name="departamentos", records=[1, 2, 3, 4])

# Paso 2: Definir variables de decisión

# Variables de producción
x = gp.Variable(m, name="x", domain=productos,type="Positive")

# Variables de faltantes
s = gp.Variable(m, name="s", domain=productos, type="Positive")

# Paso 3: Función objetivo
objetivo = 30*x[1] + 40*x[2] + 20*x[3] + 10*x[4] - 15*s[1] - 20*s[2] - 10*s[3] - 8*s[4]

# Paso 4: Restricciones

# Restricciones de capacidad
# Restriccion 1 departamento de corte
restriccion_corte = gp.Equation(m, name="restriccion_corte")
restriccion_corte[...] = 0.30*x[1] + 0.30*x[2] + 0.25*x[3] + 0.15*x[4] <= 1000

# Restricción 2 departamento de aislamiento
restriccion_aislamiento = gp.Equation(m, name="restriccion_aislamiento")
restriccion_aislamiento[...] = 0.25*x[1] + 0.35*x[2] + 0.30*x[3] + 0.10*x[4] <= 1000

# Restricción 3 departamento de costura
restriccion_costura = gp.Equation(m, name="restriccion_costura")
restriccion_costura[...] = 0.45*x[1] + 0.50*x[2] + 0.40*x[3] + 0.22*x[4] <= 1000

# Restricción 4 departamento de empaque
restriccion_empaque = gp.Equation(m, name="restriccion_empaque")
restriccion_empaque[...] = 0.15*x[1] + 0.15*x[2] + 0.10*x[3] + 0.05*x[4] <= 1000

# Restricciones de demanda
# Balance de demanda de ABRIGOS
balance_abrigos = gp.Equation(m, name="balance_abrigos")
balance_abrigos[...] = x[1] + s[1] == 800

# Balance de demanda de CHAQUETAS
balance_chaquetas = gp.Equation(m, name="balance_chaquetas")
balance_chaquetas[...] = x[2] + s[2] == 750

# Balance de demanda de PANTALONES
balance_pantalones = gp.Equation(m, name="balance_pantalones")
balance_pantalones[...] = x[3] + s[3] == 600

# Balance de demanda de GUANTES
balance_guantes = gp.Equation(m, name="balance_guantes")
balance_guantes[...] = x[4] + s[4] == 500

# Paso 6: Crear lista de todas las ecuaciones
todas_las_restricciones = [
    restriccion_corte,
    restriccion_aislamiento,
    restriccion_costura,
    restriccion_empaque,
    balance_abrigos,
    balance_chaquetas,
    balance_pantalones,
    balance_guantes
]

modelo = gp.Model(
    m,
    name="ProduccionExplicita",
    equations=todas_las_restricciones,
    sense=gp.Sense.MAX,
    problem="LP",
    objective=objetivo
)

print("\n🔄 Resolviendo modelo...")
modelo.solve()


🔄 Resolviendo modelo...


,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,Normal,OptimalGlobal,64625.0,9,9,LP,CPLEX,0.003
